In [4]:
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, END
#from langchain_community.tools import tool

from typing import List, Annotated, TypedDict, Union
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage


import operator
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from tavily import TavilyClient


In [48]:
import os

In [49]:
tool = TavilySearchResults(max_results=4) #increased number of results
print(type(tool))
print(tool.name)

<class 'langchain_community.tools.tavily_search.tool.TavilySearchResults'>
tavily_search_results_json


# Agent

In [43]:
class AgentState(TypedDict):
    messages: Annotated[
        List[Union[HumanMessage, AIMessage, SystemMessage, BaseMessage]],
        "add_message"
    ]

In [44]:
class Agent:

    def __init__(self, model, tools, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_llm)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges(
            "llm",
            self.exists_action,
            {True: "action", False: END}
        )
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile()
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def call_llm(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            if not t['name'] in self.tools:      # check for bad tool name from LLM
                print("\n ....bad tool name....")
                result = "bad tool name, retry"  # instruct LLM to retry if bad
            else:
                result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [50]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""

model = ChatOllama(model="mistral:7b-instruct")  
abot = Agent(model, [tool], system=prompt)

In [ ]:
messages = [HumanMessage(content="What is the weather in sf?")]
result = abot.graph.invoke({"messages": messages})
print(result)

In [29]:
from langchain_core.tools import tool

@tool
def calculate(what: str) -> str:
    """Make a calculation from a string expression like '2 + 2'."""
    try:
        return str(eval(what))
    except Exception as e:
        return f"Error: {e}"

@tool
def average_dog_weight(name: str) -> str:
    """Return the average weight of a given dog breed."""
    name = name.lower()
    if "scottish terrier" in name:
        return "Scottish Terriers average 20 lbs"
    elif "border collie" in name:
        return "A Border Collie's average weight is 37 lbs"
    elif "toy poodle" in name:
        return "A Toy Poodle's average weight is 7 lbs"
    else:
        return "An average dog weighs 50 lbs"

# Register tools
toolsSet = [calculate, average_dog_weight]
tool_names = {tool.name: tool for tool in toolsSet}

In [30]:
tool_names

{'calculate': StructuredTool(name='calculate', description="Make a calculation from a string expression like '2 + 2'.", args_schema=<class 'langchain_core.utils.pydantic.calculate'>, func=<function calculate at 0x00000266F90E6340>),
 'average_dog_weight': StructuredTool(name='average_dog_weight', description='Return the average weight of a given dog breed.', args_schema=<class 'langchain_core.utils.pydantic.average_dog_weight'>, func=<function average_dog_weight at 0x00000266F90E5F80>)}

# Agent Search

In [6]:
# run search
result = client.search("What is in Nvidia's new Blackwell GPU?",
                       include_answer=True)

# print the answer
result["answer"]


'The new Blackwell GPU features enhanced Transformer Engine, fifth-generation NVLink, and up to 576 GPU scaling. It has 72GB GDDR7 memory and supports advanced AI capabilities.'

In [7]:
# choose location (try to change to your own city!)

city = "San Francisco"

query = f"""
    what is the current weather in {city}?
    Should I travel there today?
    "weather.com"
"""

In [10]:
import requests
from bs4 import BeautifulSoup
from duckduckgo_search import DDGS
import re

ddg = DDGS()

def search(query, max_results=6):
    try:
        results = ddg.text(query, max_results=max_results)
        return [i["href"] for i in results]
    except Exception as e:
        print(f"returning previous results due to exception reaching ddg.")
        results = [ # cover case where DDG rate limits due to high deeplearning.ai volume
            "https://weather.com/weather/today/l/USCA0987:1:US",
            "https://weather.com/weather/hourbyhour/l/54f9d8baac32496f6b5497b4bf7a277c3e2e6cc5625de69680e6169e7e38e9a8",
        ]
        return results  


for i in search(query):
    print(i)

C:\Users\LENOVO T14s\AppData\Local\Temp\ipykernel_23116\2852339428.py:6: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  ddg = DDGS()


https://zhidao.baidu.com/question/434865041797516004.html
https://zhidao.baidu.com/question/10035040.html
https://zhidao.baidu.com/question/552230142.html
https://zhidao.baidu.com/question/361041744.html
https://forum.wordreference.com/threads/the-article-is-current.3336908/
https://forum.wordreference.com/threads/current-and-present.391667/


In [11]:
def scrape_weather_info(url):
    """Scrape content from the given URL"""
    if not url:
        return "Weather information could not be found."
    
    # fetch data
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        return "Failed to retrieve the webpage."

    # parse result
    soup = BeautifulSoup(response.text, 'html.parser')
    return soup


In [12]:
# use DuckDuckGo to find websites and take the first result
url = search(query)[0]

# scrape first wesbsite
soup = scrape_weather_info(url)

print(f"Website: {url}\n\n")
print(str(soup.body)[:50000]) # limit long outputs
# extract text
weather_data = []
for tag in soup.find_all(['h1', 'h2', 'h3', 'p']):
    text = tag.get_text(" ", strip=True)
    weather_data.append(text)

# combine all elements into a single string
weather_data = "\n".join(weather_data)

# remove all spaces from the combined text
weather_data = re.sub(r'\s+', ' ', weather_data)
    
print(f"Website: {url}\n\n")
print(weather_data)

Website: https://zhidao.baidu.com/question/434865041797516004.html


<body>
<div class="timeout hide-callback">
<div class="timeout-img"></div>
<div class="timeout-title">ç½ç»ä¸ç»åï¼è¯·ç¨åéè¯</div>
<button class="timeout-button" type="button">è¿åé¦é¡µ</button>
</div>
<div class="timeout-feedback hide-callback">
<div class="timeout-feedback-icon"></div>
<p class="timeout-feedback-title">é®é¢åé¦</p>
</div>
<script src="https://ppui-static-wap.cdn.bcebos.com/static/touch/js/mkdjump_v2_21d1ae1.js"></script>
</body>
Website: https://zhidao.baidu.com/question/434865041797516004.html


é®é¢åé¦


In [13]:
# run search
result = client.search(query, max_results=1)

# print first result
data = result["results"][0]["content"]

print(data)
import json
from pygments import highlight, lexers, formatters

# parse JSON
parsed_json = json.loads(data.replace("'", '"'))

# pretty print JSON with syntax highlighting
formatted_json = json.dumps(parsed_json, indent=4)
colorful_json = highlight(formatted_json,
                          lexers.JsonLexer(),
                          formatters.TerminalFormatter())

print(colorful_json)


{'location': {'name': 'San Francisco', 'region': 'California', 'country': 'United States of America', 'lat': 37.775, 'lon': -122.4183, 'tz_id': 'America/Los_Angeles', 'localtime_epoch': 1762643848, 'localtime': '2025-11-08 15:17'}, 'current': {'last_updated_epoch': 1762643700, 'last_updated': '2025-11-08 15:15', 'temp_c': 21.1, 'temp_f': 70.0, 'is_day': 1, 'condition': {'text': 'Partly cloudy', 'icon': '//cdn.weatherapi.com/weather/64x64/day/116.png', 'code': 1003}, 'wind_mph': 6.0, 'wind_kph': 9.7, 'wind_degree': 350, 'wind_dir': 'N', 'pressure_mb': 1015.0, 'pressure_in': 29.98, 'precip_mm': 0.0, 'precip_in': 0.0, 'humidity': 59, 'cloud': 50, 'feelslike_c': 21.1, 'feelslike_f': 70.0, 'windchill_c': 19.8, 'windchill_f': 67.6, 'heatindex_c': 20.1, 'heatindex_f': 68.2, 'dewpoint_c': 12.5, 'dewpoint_f': 54.5, 'vis_km': 16.0, 'vis_miles': 9.0, 'uv': 1.3, 'gust_mph': 10.7, 'gust_kph': 17.2}}
{
    "location": {
        "name": "San Francisco",
        "region": "California",
        "countr